In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l1_l2

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Thư viện khác
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

TensorFlow version: 2.16.2
Keras version: 3.10.0


In [17]:
import pandas as pd
import numpy as np

def load_and_prepare_data_from_table():
    """
    Trực tiếp tạo DataFrame từ dữ liệu bảng đã cho
    """
    # Dữ liệu từ bảng
    data = {
        'Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS)': [3006.89, 3356.78, 3139.87, 3348.91, 3508.93, 3470.44, 3425.85, 3647.47, 3682.88, 3653.62, 3830.21, 3562.52, 3545.33, 3542.12, 3426.63, 3245.17, 3240.56, 3281.41, 3119.38],
        'Giá trị sổ sách của cổ phiếu (BVPS)': [19253, 20377, 19181, 19576.93, 20794, 16705, 17466, 18028, 18909, 17846, 18792, 19090, 19883, 16721, 17448, 17730, 18505, 17496, 18071],
        'Chỉ số giá thị trường trên thu nhập (P/E)': [15.45, 12.96, 15.24, 15.5, 17.56, 11.93, 12.25, 10.82, 10.4, 13.62, 14.49, 15.21, 15.82, 11.29, 13.13, 17.75, 16.66, 16.36, 17.28],
        'Chỉ số giá thị trường trên giá trị sổ sách (P/B)': [2.41, 2.13, 2.49, 2.65, 2.96, 2.48, 2.4, 2.19, 2.03, 2.79, 2.95, 2.84, 2.82, 2.39, 2.58, 3.25, 2.92, 3.07, 2.98],
        'Chỉ số giá thị trường trên doanh thu thuần (P/S)': [8.41, 7.32, 4.97, 7.38, 9.88, 9.2, 7.97, 6.35, 7.51, 8.99, 10.1, 9.13, 10.94, 8.64, 9.73, 11.28, 11.45, 10.59, 11.58],
        'Tỷ suất cổ tức': [3, 3, 3, 3, 0, 0, 0, 0, 5, 4, 4, 4, 0, 0, 0, 0, 0, 4, 4],
        'Beta': [0.12, 0.11, 0.1, 0.09, 0.08, 0.29, 0.32, 0.34, 0.27, 0.29, 0.26, 0.26, 0.24, 0.11, 0.02, 0.06, 0.13, 0.41, 0.41],
        'Giá trị doanh nghiệp trên lợi nhuận trước thuế và lãi vay (EV/EBIT)': [50.91, 36.19, 53.89, 46.43, 56.65, 43.35, 44.47, 31.71, 34.19, 41.42, 53.64, 58.82, 52.37, 41.64, 44.8, 71.58, 50.15, 44.26, 68.69],
        'Tỷ suất lợi nhuận gộp biên': [40.89, 47.03, 34.81, 40.86, 47.93, 50.09, 51.1, 48.51, 50, 49.66, 48.55, 45.51, 49.11, 49.22, 49.55, 45.3, 51.75, 51.08, 42.6],
        'Tỷ lệ lãi EBIT': [16.35, 22.32, 9.42, 16.05, 17.58, 21.45, 18.17, 20.16, 22.31, 22.2, 18.92, 15.48, 20.92, 20.3, 20.61, 15.02, 21.83, 22.73, 16.56],
        'Tỷ lệ lãi EBITDA': [19.68, 25.04, 11.13, 19.28, 20.83, 24.4, 21.06, 23.28, 25.31, 25.23, 21.91, 18.01, 24.21, 22.97, 23.62, 17.7, 24.64, 25.63, 19.69],
        'Tỷ suất sinh lợi trên doanh thu thuần': [13.2, 18.01, 7.43, 12.07, 14.27, 17.66, 13.7, 17.03, 17.92, 17.35, 16.34, 13.31, 17.5, 16.72, 17.33, 12.69, 18.37, 19.04, 13.63],
        'Tỷ suất lợi nhuận trên vốn chủ sở hữu bình quân (ROEA)': [3.79, 5.3, 3.73, 4.33, 4.36, 4.83, 4.2, 5.93, 4.93, 5.2, 4.88, 4.16, 4.59, 4.73, 4.68, 3.68, 4.77, 5.36, 3.58],
        'Tỷ suất sinh lợi trên vốn dài hạn bình quân (ROCE)': [4.21, 5.9, 4.27, 5.2, 4.87, 5.34, 5.1, 6.43, 5.6, 6.09, 5.2, 4.44, 5.02, 5.29, 5.16, 4.03, 5.24, 5.92, 3.96],
        'Tỷ suất sinh lợi trên tổng tài sản bình quân (ROAA)': [2.59, 3.73, 2.64, 3.07, 3.22, 3.59, 3.14, 4.35, 3.52, 3.73, 3.58, 3.06, 3.37, 3.59, 3.59, 2.76, 3.61, 3.93, 2.49],
        'Tăng trưởng  doanh thu thuần': [-23.99, 21.49, 99.37, 13.68, 24.15, 8.4, -28.86, 14.88, 6.42, 22.89, 4.41, -4.43, 0.52, 4.58, 5.21, 7.52, 14.92, 9.58, 1.59],
        'Tăng trưởng  lợi nhuận gộp': [-20.15, 49.12, 59.56, 24.76, 45.53, 15.44, 4.44, 36.35, 11.02, 21.84, -0.8, -10.34, -1.27, 3.67, 7.38, 7.03, 21.09, 13.71, -12.66],
        'Tăng trưởng lợi nhuận trước thuế': [-7.77, 45.54, -8.3, 63.12, 34.84, 4.62, 38.02, 41.94, 32.85, 23.3, 6.89, -26.62, -5.41, -2.6, 15.64, 4.82, 20.67, 23.44, -19.14],
        'Tăng trưởng lợi nhuận sau thuế của CĐ công ty mẹ': [-7.07, 48.54, -15.33, 47.79, 34.18, 6.32, 31.24, 62.08, 33.68, 20.69, 24.48, -25.3, -1.88, 0.78, 11.61, 2.49, 20.64, 24.79, -20.14],
        'Tăng trưởng tổng tài sản': [-4.62, 0.08, 12.67, 6.09, 7.12, 13.88, 7.27, 22.07, 22.91, 10.25, 7.76, 4.92, 0.17, 9.37, 13.45, 13.8, 11.56, 16.53, 13.9],
        'Tăng trưởng nợ dài hạn': [14.2, 14.33, 9.99, 20.7, 3.64, -0.86, -0.94, 8.02, 3.16, 4.04, 5.11, 2.19, 2.2, 3.72, 3.96, 3.5, 3.55, 3.48, 53.62],
        'Tăng trưởng nợ phải trả': [-20.34, -15.27, 31.47, -11.55, -15.01, 8.46, -14.56, 29.28, 37.59, 20.62, 8.83, 2.91, -11.91, -10.02, 5.81, 8.34, -2.18, 56.55, 45.12],
        'Tăng trưởng vốn chủ sở hữu': [5.49, 7.48, 5.93, 15.6, 18.8, 17.23, 18.37, 19.71, 18.22, 6.83, 7.59, 5.89, 5.15, 17.12, 16.06, 16.1, 16.34, 4.64, 4.61],
        'Tăng trưởng vốn điều lệ': [np.nan, 9.99, 9.99, 9.99, 9.99, 29.98, 29.98, 29.98, 29.98, np.nan, np.nan, np.nan, np.nan, np.nan, 24.99, 24.99, 24.99, 24.99, 1],
        'Tỷ số thanh toán bằng tiền mặt': [0.43, 0.31, 0.19, 0.23, 0.24, 0.15, 0.11, 0.14, 0.26, 0.06, 0.12, 0.21, 0.21, 0.39, 0.7, 0.68, 0.75, 0.52, 0.41],
        'Tỷ số thanh toán nhanh': [1.9, 2.2, 1.9, 2.45, 2.48, 2.29, 2.44, 1.98, 1.91, 1.6, 1.98, 1.85, 2.13, 2.26, 2.41, 2.11, 2.73, 1.84, 1.97],
        'Tỷ số thanh toán nhanh (Đã loại trừ HTK, Phải thu ngắn hạn - Tham khảo)': [0.67, 0.59, 0.37, 0.71, 0.73, 0.72, 0.78, 0.68, 0.74, 0.37, 0.62, 0.74, 0.73, 0.77, 1.05, 0.95, 1.15, 0.84, 1],
        'Tỷ số thanh toán hiện hành (ngắn hạn)': [2.52, 3.15, 2.69, 3.32, 3.33, 3.3, 3.44, 3.09, 3.01, 2.94, 3.41, 3.01, 3.44, 3.74, 3.72, 3.21, 4.06, 2.58, 2.69],
        'Khả năng thanh toán lãi vay': [91.65, 222.58, 135.35, 581.07, 1023.53, 1763.34, 791.48, 54.44, 57.87, 32.35, 54.89, 54.34, 71.53, 77.73, 109.09, 72.74, 133.78, 137.43, 54.34],
        'Vòng quay phải thu khách hàng': [0.68, 0.71, 1.11, 0.77, 0.71, 0.64, 0.72, 0.84, 0.71, 0.79, 0.82, 0.9, 0.78, 0.9, 0.9, 0.97, 0.86, 0.91, 0.87],
        'Thời gian thu tiền khách hàng bình quân': [536.13, 514.42, 327.4, 473.38, 516.17, 572.58, 508.88, 435.03, 515.75, 459.21, 445.79, 407.41, 465.91, 405.37, 403.9, 374.75, 424.9, 400.26, 419.74],
        'Vòng quay hàng tồn kho': [0.77, 0.67, 1.25, 0.85, 0.75, 0.59, 0.6, 0.62, 0.41, 0.41, 0.41, 0.49, 0.4, 0.44, 0.44, 0.53, 0.44, 0.51, 0.61],
        'Thời gian tồn kho bình quân': [470.98, 545.99, 290.97, 429.23, 486.72, 613.9, 608.15, 587.4, 891.18, 891.68, 895.21, 745.42, 904.79, 822.88, 833.96, 692.68, 834.22, 722.19, 600.64],
        'Vòng quay phải trả nhà cung cấp': [1.51, 1.35, 3.1, 2.28, 1.99, 1.54, 1.78, 1.9, 1.25, 1.51, 1.93, 2.31, 1.69, 2, 2.11, 2.06, 1.67, 2.44, 2.02],
        'Thời gian trả tiền khách hàng bình quân': [242.03, 270.2, 117.79, 160.05, 183.53, 237.21, 205.24, 192.48, 291.22, 241.75, 189.17, 158.08, 215.77, 182.24, 173.22, 177.32, 219.16, 149.74, 180.25],
        'Vòng quay tài sản cố định (Hiệu suất sử dụng tài sản cố định)': [1.18, 1.31, 2.4, 1.7, 1.48, 1.44, 1.77, 1.62, 1.09, 1.22, 1.25, 1.3, 1.07, 1.22, 1.24, 1.41, 1.29, 1.32, 1.2],
        'Vòng quay tổng tài sản (Hiệu suất sử dụng toàn bộ tài sản)': [0.19, 0.21, 0.36, 0.25, 0.23, 0.2, 0.23, 0.26, 0.2, 0.21, 0.22, 0.23, 0.19, 0.21, 0.21, 0.22, 0.2, 0.21, 0.18],
        'Vòng quay vốn chủ sở hữu': [0.29, 0.3, 0.51, 0.36, 0.31, 0.28, 0.31, 0.35, 0.28, 0.3, 0.3, 0.31, 0.26, 0.28, 0.27, 0.29, 0.26, 0.28, 0.26],
        'Tỷ số Nợ ngắn hạn trên Tổng nợ phải trả': [74.83, 70.02, 77.25, 70.1, 69.3, 72.59, 73.62, 75.02, 76.98, 76.36, 74.52, 75.19, 73.29, 72.75, 74.97, 76.3, 71.73, 81.99, 73.5],
        'Tỷ số Nợ vay trên Tổng tài sản': [8.72, 6.3, 8.12, 6.25, 5.88, 4.96, 4.58, 4.01, 8.06, 5.84, 3.4, 3.98, 4.09, 2.54, 2.25, 2.82, 1.8, 2.73, 7.34],
        'Tỷ số Nợ trên Tổng tài sản': [32.07, 27.08, 31.08, 26.93, 25.45, 25.79, 24.76, 28.52, 28.49, 28.22, 25, 27.98, 25.05, 23.21, 23.32, 26.63, 21.96, 31.19, 29.71],
        'Tỷ số Vốn chủ sở hữu trên Tổng tài sản': [66.76, 71.66, 67.84, 72.55, 74.04, 73.77, 74.86, 71.15, 71.21, 71.48, 74.74, 71.81, 74.75, 76.54, 76.46, 73.26, 77.95, 68.73, 70.22],
        'Tỷ số Nợ ngắn hạn trên Vốn chủ sở hữu': [74.83, 70.02, 77.25, 70.1, 69.3, 72.59, 73.62, 75.02, 30.79, 30.14, 24.93, 29.29, 24.56, 22.06, 22.86, 27.74, 20.21, 37.2, 31.1],
        'Tỷ số Nợ vay trên Vốn chủ sở hữu': [13.06, 8.8, 11.97, 8.61, 7.94, 6.72, 6.12, 5.63, 11.31, 8.17, 4.55, 5.55, 5.47, 3.32, 2.94, 3.85, 2.31, 3.97, 10.45],
        'Tỷ số Nợ trên Vốn chủ sở hữu': [48.04, 37.79, 45.82, 37.12, 34.37, 34.96, 33.07, 40.09, 40, 39.48, 33.45, 38.96, 33.51, 30.33, 30.5, 36.36, 28.18, 45.37, 42.31],
        'Tỷ số dòng tiền HĐKD trên doanh thu thuần': [0.48, 16.98, -10.01, 26.37, 8.19, 11.41, 11.1, 6.11, -4.9, 11.73, 35.13, 22.56, 4.56, 12.67, np.nan, np.nan, 0.49, np.nan, np.nan],
        'Khả năng chi trả nợ ngắn hạn từ dòng tiền HĐKD': [0.38, 18.7, -14.19, 36.41, 10.3, 12.12, 13.74, 7, -4.29, 12.06, 41.22, 23.95, 4.79, 15.89, np.nan, np.nan, 0.61, np.nan, np.nan],
        'Khả năng chi trả nợ ngắn hạn từ lưu chuyển tiền thuần trong kỳ': [13.48, -24.89, -3.38, -1.93, -0.12, -6.46, -3.91, 4.82, 13.43, -21.45, 4.74, 11.01, -3.5, 17.19, np.nan, np.nan, -14.34, np.nan, np.nan],
        'Tỷ lệ dồn tích (Phương pháp Cân đối kế toán)': [4.92, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
        'Tỷ lệ dồn tích (Phương pháp Dòng tiền)': [2.28, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
        'Dòng tiền từ HĐKD trên Tổng tài sản': [0.09, 3.55, -3.41, 6.87, 1.82, 2.27, 2.5, 1.5, -0.94, 2.6, 7.68, 5.04, 0.88, 2.68, np.nan, np.nan, 0.1, np.nan, np.nan],
        'Dòng tiền từ HĐKD trên Vốn chủ sở hữu': [0.14, 4.95, -5.02, 9.47, 2.45, 3.08, 3.35, 2.11, -1.32, 3.64, 10.28, 7.02, 1.18, 3.51, np.nan, np.nan, 0.12, np.nan, np.nan],
        'Dòng tiền từ HĐKD trên Lợi nhuận thuần từ HĐKD': [2.98, 76.53, -107.02, 168.47, 46.7, 53.2, 61.27, 30.99, -22.36, 54.63, 183.94, 148.25, 22.06, 63.13, np.nan, np.nan, 2.2, np.nan, np.nan],
        'Khả năng thanh toán nợ từ dòng tiền HĐKD': [0.29, 13.09, -10.96, 25.52, 7.14, 8.8, 10.12, 5.25, -3.3, 9.21, 30.72, 18.01, 3.51, 11.56, np.nan, np.nan, 0.44, np.nan, np.nan],
        'Dòng tiền từ HĐKD trên mỗi cổ phần (CPS)': [26.61, 1008.37, -963.27, 1854.47, 510.33, 513.73, 584.41, 379.53, -249.92, 648.75, 1930.89, 1339.57, 233.9, 586.35, np.nan, np.nan, 22.88, np.nan, np.nan],
        'Giá vốn hàng bán/Doanh thu thuần': [59.11, 52.97, 65.19, 59.14, 52.07, 49.91, 48.9, 51.49, 50, 50.34, 51.45, 54.49, 50.89, 50.78, 50.45, 54.7, 48.25, 48.92, 57.4],
        'Chi phí bán hàng/Doanh thu thuần': [20.5, 19.64, 16.28, 16.3, 24.9, 22.63, 24.87, 20.43, 23.77, 24.11, 24.42, 19, 24.1, 23.29, 25.56, 20.97, 21.07, 17.31, 20.04],
        'Chi phí quản lý doanh nghiệp/Doanh thu thuần': [5.47, 6.3, 8.7, 11.22, 6.71, 7.83, 8.12, 11.56, 6.03, 5.01, 5.92, 13.77, 6.57, 4.78, 4.84, 11.7, 10.68, 12.18, 5.36],
        'Chi phí lãi vay/Doanh thu thuần': [0.18, 0.1, 0.07, 0.03, 0.02, 0.01, 0.02, 0.37, 0.39, 0.69, 0.34, 0.28, 0.29, 0.26, 0.19, 0.21, 0.16, 0.17, 0.3],
        'Tài sản ngắn hạn/Tổng tài sản': [60.57, 59.7, 64.52, 62.59, 58.8, 61.76, 62.74, 66.07, 66.09, 63.28, 63.49, 63.4, 63.16, 63.14, 64.96, 65.2, 64.03, 65.98, 58.67],
        'Tiền/Tài sản ngắn hạn': [17.23, 9.83, 7.06, 7.04, 7.25, 4.63, 3.29, 4.44, 8.69, 2.13, 3.5, 6.96, 5.97, 10.41, 18.88, 21.28, 18.56, 19.99, 15.42],
        'Đầu tư tài chính ngắn hạn/Tài sản ngắn hạn': [9.12, 8.95, 6.62, 14.32, 14.64, 17.15, 19.5, 17.67, 15.78, 10.35, 14.82, 17.54, 15.16, 10.09, 9.39, 8.43, 9.71, 12.71, 21.88],
        'Phải thu ngắn hạn/Tài sản ngắn hạn': [48.45, 50.76, 56.37, 51.82, 51.69, 46.66, 47.36, 41.08, 38.14, 40.87, 38.73, 35.91, 39.88, 39.05, 35.83, 35.24, 38.1, 37.5, 34.67],
        'Hàng tồn kho/Tài sản ngắn hạn': [24.71, 30.01, 29.42, 26.13, 25.68, 30.72, 29.25, 35.95, 36.68, 45.39, 41.98, 38.67, 38.21, 39.6, 35.1, 34.23, 32.87, 28.72, 26.6],
        'Tài sản ngắn hạn khác/Tài sản ngắn hạn': [0.49, 0.45, 0.53, 0.69, 0.75, 0.84, 0.59, 0.87, 0.71, 1.27, 0.98, 0.92, 0.79, 0.84, 0.79, 0.82, 0.75, 1.07, 1.43],
        'Tài sản dài hạn/Tổng tài sản': [39.43, 40.3, 35.48, 37.41, 41.2, 38.24, 37.26, 33.93, 33.91, 36.72, 36.51, 36.6, 36.84, 36.86, 35.04, 34.8, 35.97, 34.02, 41.33],
        'Tài sản cố định/Tổng tài sản': [15.94, 15.63, 14.03, 15.91, 14.73, 13.47, 12.43, 18.82, 17.35, 17.89, 17.24, 17.96, 17.93, 17.35, 16.03, 14.91, 15.6, 15.6, 14.88],
        'Tài sản cố định hữu hình/Tài sản cố định': [74.56, 73.92, 73.31, 75.55, 74.47, 73.51, 72.46, 83.16, 82.77, 82.45, 81.9, 83.71, 83.81, 83.82, 83.36, 83.17, 83.42, 77.69, 77.19],
        'Tài sản thuê tài chính/Tài sản cố định': [0, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
        'Tài sản vô hình/Tài sản cố định': [25.44, 26.08, 26.69, 24.45, 25.53, 26.49, 27.54, 16.84, 17.23, 17.55, 18.1, 16.29, 16.19, 16.18, 16.64, 16.83, 16.58, 22.31, 22.81],
        'XDCBDD/Tài sản cố định': [71.39, 76.48, 79.37, 64.59, 96.76, 97.56, 107.69, 25.73, 39.09, 45.12, 52.3, 45.55, 46.56, 52.41, 62.06, 69.16, 68.64, 58.08, 122.13]
    }
    
    # Tạo DataFrame
    df = pd.DataFrame(data)
    
    # Tạo index thời gian
    quarters = pd.date_range(start='2021-03-31', periods=19, freq='Q')
    df.index = quarters
    df['Quarter'] = df.index
    
    print(f"Data shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    
    return df

# Load dữ liệu trực tiếp
df = load_and_prepare_data_from_table()
print("\nData loaded successfully!")
print(f"Shape: {df.shape}")

Data shape: (19, 72)

Columns: ['Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS)', 'Giá trị sổ sách của cổ phiếu (BVPS)', 'Chỉ số giá thị trường trên thu nhập (P/E)', 'Chỉ số giá thị trường trên giá trị sổ sách (P/B)', 'Chỉ số giá thị trường trên doanh thu thuần (P/S)', 'Tỷ suất cổ tức', 'Beta', 'Giá trị doanh nghiệp trên lợi nhuận trước thuế và lãi vay (EV/EBIT)', 'Tỷ suất lợi nhuận gộp biên', 'Tỷ lệ lãi EBIT', 'Tỷ lệ lãi EBITDA', 'Tỷ suất sinh lợi trên doanh thu thuần', 'Tỷ suất lợi nhuận trên vốn chủ sở hữu bình quân (ROEA)', 'Tỷ suất sinh lợi trên vốn dài hạn bình quân (ROCE)', 'Tỷ suất sinh lợi trên tổng tài sản bình quân (ROAA)', 'Tăng trưởng  doanh thu thuần', 'Tăng trưởng  lợi nhuận gộp', 'Tăng trưởng lợi nhuận trước thuế', 'Tăng trưởng lợi nhuận sau thuế của CĐ công ty mẹ', 'Tăng trưởng tổng tài sản', 'Tăng trưởng nợ dài hạn', 'Tăng trưởng nợ phải trả', 'Tăng trưởng vốn chủ sở hữu', 'Tăng trưởng vốn điều lệ', 'Tỷ số thanh toán bằng tiền mặt', 'Tỷ số thanh toán nhanh', 'Tỷ s

In [18]:
df.head()

,Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS),Giá trị sổ sách của cổ phiếu (BVPS),Chỉ số giá thị trường trên thu nhập (P/E),Chỉ số giá thị trường trên giá trị sổ sách (P/B),Chỉ số giá thị trường trên doanh thu thuần (P/S),Tỷ suất cổ tức,Beta,Giá trị doanh nghiệp trên lợi nhuận trước thuế và lãi vay (EV/EBIT),Tỷ suất lợi nhuận gộp biên,Tỷ lệ lãi EBIT,...,Phải thu ngắn hạn/Tài sản ngắn hạn,Hàng tồn kho/Tài sản ngắn hạn,Tài sản ngắn hạn khác/Tài sản ngắn hạn,Tài sản dài hạn/Tổng tài sản,Tài sản cố định/Tổng tài sản,Tài sản cố định hữu hình/Tài sản cố định,Tài sản thuê tài chính/Tài sản cố định,Tài sản vô hình/Tài sản cố định,XDCBDD/Tài sản cố định,Quarter
2021-03-31,3006.89,19253.00,15.45,2.41,8.41,3,0.12,50.91,40.89,16.35,...,48.45,24.71,0.49,39.43,15.94,74.56,0.0,25.44,71.39,2021-03-31
2021-06-30,3356.78,20377.00,12.96,2.13,7.32,3,0.11,36.19,47.03,22.32,...,50.76,30.01,0.45,40.30,15.63,73.92,NaN,26.08,76.48,2021-06-30
2021-09-30,3139.87,19181.00,15.24,2.49,4.97,3,0.10,53.89,34.81,9.42,...,56.37,29.42,0.53,35.48,14.03,73.31,NaN,26.69,79.37,2021-09-30
2021-12-31,3348.91,19576.93,15.50,2.65,7.38,3,0.09,46.43,40.86,16.05,...,51.82,26.13,0.69,37.41,15.91,75.55,NaN,24.45,64.59,2021-12-31
2022-03-31,3508.93,20794.00,17.56,2.96,9.88,0,0.08,56.65,47.93,17.58,...,51.69,25.68,0.75,41.20,14.73,74.47,NaN,25.53,96.76,2022-03-31


In [19]:
class LSTMPreprocessor:
    """Tiền xử lý dữ liệu cho mô hình LSTM"""
    
    def __init__(self, target_col='Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS)'):
        self.target_col = target_col
        self.scalers = {}
        self.selected_features = None
        
    def select_features(self, df, correlation_threshold=0.6):
        """Chọn features dựa trên correlation với target"""
        # Tính correlation
        corr_matrix = df.corr()
        target_corr = corr_matrix[self.target_col].abs().sort_values(ascending=False)
        
        # Chọn features có correlation cao
        high_corr_features = target_corr[target_corr > correlation_threshold].index.tolist()
        
        # Loại bỏ target khỏi features
        if self.target_col in high_corr_features:
            high_corr_features.remove(self.target_col)
            
        # Thêm một số features quan trọng dựa trên domain knowledge
        important_features = [
            'Tỷ suất lợi nhuận gộp biên',
            'Tỷ lệ lãi EBIT',
            'Tăng trưởng doanh thu thuần',
            'Vòng quay tổng tài sản',
            'Tỷ số Nợ trên Tổng tài sản',
            'Beta',
            'Giá trị sổ sách của cổ phiếu (BVPS)',
            'Tỷ suất sinh lợi trên vốn chủ sở hữu bình quân (ROEA)'
        ]
        
        # Kết hợp features
        all_features = list(set(high_corr_features + important_features))
        
        # Chỉ lấy features có trong df
        self.selected_features = [f for f in all_features if f in df.columns]
        
        print(f"Selected {len(self.selected_features)} features:")
        for f in self.selected_features[:10]:
            print(f"  - {f}")
        
        return self.selected_features
    
    def create_sequences(self, data, sequence_length=4):
        """Tạo sequences cho LSTM"""
        X, y = [], []
        
        for i in range(len(data) - sequence_length):
            # Input sequence
            X_seq = data.iloc[i:i+sequence_length].values
            
            # Target (giá trị tiếp theo của EPS)
            y_val = data.iloc[i+sequence_length][self.target_col]
            
            X.append(X_seq)
            y.append(y_val)
        
        return np.array(X), np.array(y)
    
    def prepare_train_test(self, df, test_size=0.2, sequence_length=4):
        """Chuẩn bị train/test split cho time series"""
        # Chọn features
        if self.selected_features is None:
            self.selected_features = self.select_features(df)
        
        # Tạo dataset với features và target
        features_data = df[self.selected_features + [self.target_col]].copy()
        
        # Fill missing values
        features_data = features_data.fillna(method='ffill').fillna(method='bfill')
        
        # Chuẩn hóa dữ liệu
        scaled_data = features_data.copy()
        for col in features_data.columns:
            scaler = MinMaxScaler()
            scaled_data[col] = scaler.fit_transform(features_data[[col]])
            self.scalers[col] = scaler
        
        # Tạo sequences
        X, y = self.create_sequences(scaled_data, sequence_length)
        
        # Chia train/test (giữ nguyên thứ tự thời gian)
        split_idx = int(len(X) * (1 - test_size))
        
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        
        print(f"Train shape: X={X_train.shape}, y={y_train.shape}")
        print(f"Test shape: X={X_test.shape}, y={y_test.shape}")
        
        return X_train, X_test, y_train, y_test, features_data
    
    def inverse_transform(self, scaled_values, col_name):
        """Chuyển đổi ngược giá trị đã chuẩn hóa"""
        if col_name in self.scalers:
            return self.scalers[col_name].inverse_transform(
                scaled_values.reshape(-1, 1)
            ).flatten()
        return scaled_values

In [20]:
class LSTMModel:
    """Lớp xây dựng và huấn luyện mô hình LSTM"""
    
    def __init__(self, input_shape, model_type='simple'):
        self.input_shape = input_shape
        self.model_type = model_type
        self.model = None
        self.history = None
        
    def build_simple_lstm(self):
        """Xây dựng mô hình LSTM đơn giản"""
        model = Sequential([
            # LSTM layer 1
            LSTM(64, 
                 return_sequences=True, 
                 input_shape=self.input_shape,
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # LSTM layer 2
            LSTM(32, 
                 return_sequences=True,
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # LSTM layer 3
            LSTM(16, 
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # Dense layers
            Dense(32, activation='relu'),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1, activation='linear')  # Regression output
        ])
        
        return model
    
    def build_bidirectional_lstm(self):
        """Xây dựng mô hình Bidirectional LSTM"""
        inputs = Input(shape=self.input_shape)
        
        # Bidirectional LSTM layers
        x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
        x = Dropout(0.3)(x)
        x = BatchNormalization()(x)
        
        x = Bidirectional(LSTM(32, return_sequences=True))(x)
        x = Dropout(0.3)(x)
        x = BatchNormalization()(x)
        
        x = Bidirectional(LSTM(16))(x)
        x = Dropout(0.3)(x)
        x = BatchNormalization()(x)
        
        # Attention mechanism
        attention = Dense(1, activation='tanh')(x)
        attention = Flatten()(attention)
        attention = Activation('softmax')(attention)
        attention = RepeatVector(16)(attention)
        attention = Permute([2, 1])(attention)
        
        # Apply attention
        sent_representation = Multiply()([x, attention])
        sent_representation = Lambda(lambda xin: tf.reduce_sum(xin, axis=1))(sent_representation)
        
        # Output layers
        x = Dense(32, activation='relu')(sent_representation)
        x = Dropout(0.2)(x)
        x = Dense(16, activation='relu')(x)
        outputs = Dense(1, activation='linear')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        return model
    
    def build_conv_lstm(self):
        """Xây dựng mô hình ConvLSTM"""
        model = Sequential([
            # Conv1D layers for feature extraction
            Conv1D(filters=64, kernel_size=3, padding='same', 
                   input_shape=self.input_shape),
            BatchNormalization(),
            Activation('relu'),
            Dropout(0.2),
            
            Conv1D(filters=32, kernel_size=3, padding='same'),
            BatchNormalization(),
            Activation('relu'),
            Dropout(0.2),
            
            # LSTM layers
            LSTM(64, return_sequences=True),
            Dropout(0.3),
            BatchNormalization(),
            
            LSTM(32),
            Dropout(0.3),
            BatchNormalization(),
            
            # Dense layers
            Dense(32, activation='relu'),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1, activation='linear')
        ])
        
        return model
    
    def build_model(self):
        """Xây dựng mô hình dựa trên loại được chọn"""
        if self.model_type == 'simple':
            self.model = self.build_simple_lstm()
        elif self.model_type == 'bidirectional':
            self.model = self.build_bidirectional_lstm()
        elif self.model_type == 'conv_lstm':
            self.model = self.build_conv_lstm()
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        
        # Compile model
        optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
        self.model.compile(
            optimizer=optimizer,
            loss='huber_loss',  # Huber loss is robust to outliers
            metrics=['mae', 'mse']
        )
        
        self.model.summary()
        return self.model
    
    def train(self, X_train, y_train, X_val, y_val, epochs=200, batch_size=4):
        """Huấn luyện mô hình"""
        # Callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=50,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=20,
                min_lr=1e-6,
                verbose=1
            ),
            ModelCheckpoint(
                'best_lstm_model.keras',
                monitor='val_loss',
                save_best_only=True,
                verbose=1
            )
        ]
        
        # Train model
        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
        
        return self.history
    
    def predict(self, X):
        """Dự báo với mô hình đã huấn luyện"""
        return self.model.predict(X)
    
    def evaluate(self, X_test, y_test):
        """Đánh giá mô hình"""
        predictions = self.predict(X_test)
        
        metrics = {
            'MAE': mean_absolute_error(y_test, predictions),
            'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
            'R2': r2_score(y_test, predictions)
        }
        
        # Calculate MAPE (Mean Absolute Percentage Error)
        mape = np.mean(np.abs((y_test - predictions.flatten()) / y_test)) * 100
        metrics['MAPE'] = mape
        
        return predictions, metrics

In [26]:
# Sửa phần compile model trong class LSTMModel
class LSTMModel:
    """Lớp xây dựng và huấn luyện mô hình LSTM"""
    
    def __init__(self, input_shape, model_type='simple'):
        self.input_shape = input_shape
        self.model_type = model_type
        self.model = None
        self.history = None
        
    def build_simple_lstm(self):
        """Xây dựng mô hình LSTM đơn giản"""
        model = Sequential([
            # LSTM layer 1
            LSTM(64, 
                 return_sequences=True, 
                 input_shape=self.input_shape,
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # LSTM layer 2
            LSTM(32, 
                 return_sequences=True,
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # LSTM layer 3
            LSTM(16, 
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # Dense layers
            Dense(32, activation='relu'),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1, activation='linear')  # Regression output
        ])
        
        return model
    
    # ... các phương thức build khác giữ nguyên ...
    
    def build_model(self):
        """Xây dựng mô hình dựa trên loại được chọn"""
        if self.model_type == 'simple':
            self.model = self.build_simple_lstm()
        elif self.model_type == 'bidirectional':
            self.model = self.build_bidirectional_lstm()
        elif self.model_type == 'conv_lstm':
            self.model = self.build_conv_lstm()
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        
        # Compile model với loss function thay thế
        optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
        self.model.compile(
            optimizer=optimizer,
            loss='mse',  # Thay huber_loss bằng mse
            metrics=['mae', 'mse']
        )
        
        self.model.summary()
        return self.model

In [28]:
# Sửa phần compile model trong class LSTMModel
class LSTMModel:
    """Lớp xây dựng và huấn luyện mô hình LSTM"""
    
    def __init__(self, input_shape, model_type='simple'):
        self.input_shape = input_shape
        self.model_type = model_type
        self.model = None
        self.history = None
        
    def build_simple_lstm(self):
        """Xây dựng mô hình LSTM đơn giản"""
        model = Sequential([
            # LSTM layer 1
            LSTM(64, 
                 return_sequences=True, 
                 input_shape=self.input_shape,
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # LSTM layer 2
            LSTM(32, 
                 return_sequences=True,
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # LSTM layer 3
            LSTM(16, 
                 kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
            Dropout(0.3),
            BatchNormalization(),
            
            # Dense layers
            Dense(32, activation='relu'),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1, activation='linear')  # Regression output
        ])
        
        return model
    
    # ... các phương thức build khác giữ nguyên ...
    
    def build_model(self):
        """Xây dựng mô hình dựa trên loại được chọn"""
        if self.model_type == 'simple':
            self.model = self.build_simple_lstm()
        elif self.model_type == 'bidirectional':
            self.model = self.build_bidirectional_lstm()
        elif self.model_type == 'conv_lstm':
            self.model = self.build_conv_lstm()
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        
        # Compile model với loss function thay thế
        optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
        self.model.compile(
            optimizer=optimizer,
            loss='mse',  # Thay huber_loss bằng mse
            metrics=['mae', 'mse']
        )
        
        self.model.summary()
        return self.model

In [30]:
def run_lstm_pipeline(df, model_type='simple'):
    """Chạy toàn bộ pipeline LSTM"""
    
    # Khởi tạo preprocessor
    preprocessor = LSTMPreprocessor()
    
    # Chuẩn bị dữ liệu
    print("=" * 50)
    print("PREPARING DATA...")
    print("=" * 50)
    
    X_train, X_test, y_train, y_test, features_data = preprocessor.prepare_train_test(
        df, test_size=0.3, sequence_length=4
    )
    
    # Xây dựng mô hình
    print("\n" + "=" * 50)
    print("BUILDING MODEL...")
    print("=" * 50)
    
    input_shape = (X_train.shape[1], X_train.shape[2])
    lstm_model = LSTMModel(input_shape, model_type)
    model = lstm_model.build_model()
    
    # Huấn luyện mô hình
    print("\n" + "=" * 50)
    print("TRAINING MODEL...")
    print("=" * 50)
    
    # Chia validation từ train
    val_split = 0.2
    val_size = int(len(X_train) * val_split)
    
    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_train_final = X_train[:-val_size]
    y_train_final = y_train[:-val_size]
    
    history = lstm_model.train(
        X_train_final, y_train_final,
        X_val, y_val,
        epochs=300,
        batch_size=2
    )
    
    # Đánh giá mô hình
    print("\n" + "=" * 50)
    print("EVALUATING MODEL...")
    print("=" * 50)
    
    predictions, metrics = lstm_model.evaluate(X_test, y_test)
    
    # Inverse transform predictions
    y_test_original = preprocessor.inverse_transform(y_test, preprocessor.target_col)
    predictions_original = preprocessor.inverse_transform(predictions.flatten(), preprocessor.target_col)
    
    print("\nModel Performance Metrics:")
    print("-" * 30)
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")
    
    return {
        'model': lstm_model,
        'preprocessor': preprocessor,
        'history': history,
        'metrics': metrics,
        'y_test': y_test_original,
        'predictions': predictions_original,
        'X_test': X_test
    }

# Chạy pipeline
results = run_lstm_pipeline(df, model_type='simple')

PREPARING DATA...
Selected 8 features:
  - Hàng tồn kho/Tài sản ngắn hạn
  - Tỷ số Nợ trên Tổng tài sản
  - Tỷ lệ lãi EBIT
  - Tiền/Tài sản ngắn hạn
  - Beta
  - Tỷ suất sinh lợi trên tổng tài sản bình quân (ROAA)
  - Tỷ suất lợi nhuận gộp biên
  - Giá trị sổ sách của cổ phiếu (BVPS)
Train shape: X=(10, 4, 9), y=(10,)
Test shape: X=(5, 4, 9), y=(5,)

BUILDING MODEL...


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_9 (LSTM)                   │ (None, 4, 64)          │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 4, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 4, 32)          │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 4, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 4, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,033 (140.75 KB)

 Trainable params: 35,809 (139.88 KB)

 Non-trainable params: 224 (896.00 B)


TRAINING MODEL...


AttributeError: 'LSTMModel' object has no attribute 'train'

In [23]:
def forecast_future(model, preprocessor, df, n_steps_ahead=4):
    """Dự báo các quý tiếp theo"""
    
    # Lấy dữ liệu gần nhất
    recent_data = df[preprocessor.selected_features + [preprocessor.target_col]].tail(4)
    
    # Chuẩn hóa
    scaled_data = recent_data.copy()
    for col in recent_data.columns:
        if col in preprocessor.scalers:
            scaled_data[col] = preprocessor.scalers[col].transform(recent_data[[col]])
    
    # Tạo sequence cuối cùng
    last_sequence = scaled_data.values.reshape(1, 4, len(scaled_data.columns))
    
    # Dự báo
    forecasts = []
    current_sequence = last_sequence.copy()
    
    for step in range(n_steps_ahead):
        # Dự báo giá trị tiếp theo
        pred_scaled = model.predict(current_sequence)[0, 0]
        
        # Inverse transform
        pred_original = preprocessor.inverse_transform(
            np.array([pred_scaled]), 
            preprocessor.target_col
        )[0]
        
        forecasts.append(pred_original)
        
        # Cập nhật sequence cho bước tiếp theo
        # (Trong thực tế, cần cập nhật cả features khác)
        new_row = current_sequence[0, -1].copy()
        new_row[-1] = pred_scaled  # Cập nhật EPS dự báo
        
        # Shift sequence
        current_sequence = np.roll(current_sequence, -1, axis=1)
        current_sequence[0, -1] = new_row
    
    # Tạo timeline cho dự báo
    last_date = df.index[-1]
    forecast_dates = pd.date_range(
        start=last_date + pd.offsets.QuarterBegin(1),
        periods=n_steps_ahead,
        freq='Q'
    )
    
    return forecast_dates, forecasts

# Dự báo 4 quý tiếp theo
future_dates, future_eps = forecast_future(
    results['model'].model,
    results['preprocessor'],
    df,
    n_steps_ahead=4
)

print("\n" + "=" * 50)
print("FUTURE FORECAST")
print("=" * 50)
for date, eps in zip(future_dates, future_eps):
    print(f"{date.strftime('%Y-Q%q')}: EPS = {eps:,.2f}")

NameError: name 'results' is not defined

In [24]:
class EnsembleLSTM:
    """Ensemble nhiều mô hình LSTM"""
    
    def __init__(self, input_shape):
        self.input_shape = input_shape
        self.models = []
        
    def build_ensemble(self, n_models=5):
        """Xây dựng ensemble của nhiều mô hình"""
        for i in range(n_models):
            print(f"\nBuilding model {i+1}/{n_models}")
            
            # Tạo mô hình với kiến trúc khác nhau
            if i % 3 == 0:
                model_type = 'simple'
            elif i % 3 == 1:
                model_type = 'bidirectional'
            else:
                model_type = 'conv_lstm'
            
            lstm_model = LSTMModel(self.input_shape, model_type)
            model = lstm_model.build_model()
            
            # Thay đổi learning rate cho từng model
            optimizer = Adam(learning_rate=0.001 * (0.8 ** i))
            model.compile(
                optimizer=optimizer,
                loss='huber_loss',
                metrics=['mae']
            )
            
            self.models.append(model)
        
        return self.models
    
    def train_ensemble(self, X_train, y_train, X_val, y_val, 
                       epochs=150, batch_size=2):
        """Huấn luyện ensemble"""
        histories = []
        
        for i, model in enumerate(self.models):
            print(f"\n{'='*50}")
            print(f"Training model {i+1}/{len(self.models)}")
            print(f"{'='*50}")
            
            # Callbacks cho từng model
            callbacks = [
                EarlyStopping(patience=30, restore_best_weights=True),
                ReduceLROnPlateau(factor=0.5, patience=15)
            ]
            
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=epochs,
                batch_size=batch_size,
                callbacks=callbacks,
                verbose=0
            )
            
            histories.append(history)
            print(f"Model {i+1} - Best val loss: {min(history.history['val_loss']):.4f}")
        
        return histories
    
    def predict_ensemble(self, X):
        """Dự báo bằng ensemble (average predictions)"""
        predictions = []
        
        for model in self.models:
            pred = model.predict(X, verbose=0)
            predictions.append(pred)
        
        # Tính trung bình
        ensemble_pred = np.mean(predictions, axis=0)
        
        # Tính độ tin cậy (standard deviation)
        pred_std = np.std(predictions, axis=0)
        
        return ensemble_pred, pred_std

# Sử dụng ensemble (tùy chọn)
def run_ensemble_pipeline(df):
    """Chạy pipeline với ensemble models"""
    
    preprocessor = LSTMPreprocessor()
    X_train, X_test, y_train, y_test, _ = preprocessor.prepare_train_test(
        df, test_size=0.3, sequence_length=4
    )
    
    # Build ensemble
    input_shape = (X_train.shape[1], X_train.shape[2])
    ensemble = EnsembleLSTM(input_shape)
    ensemble.build_ensemble(n_models=3)
    
    # Train ensemble
    val_split = 0.2
    val_size = int(len(X_train) * val_split)
    
    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_train_final = X_train[:-val_size]
    y_train_final = y_train[:-val_size]
    
    histories = ensemble.train_ensemble(
        X_train_final, y_train_final,
        X_val, y_val,
        epochs=200,
        batch_size=2
    )
    
    # Predict with ensemble
    predictions, std_dev = ensemble.predict_ensemble(X_test)
    
    # Inverse transform
    y_test_original = preprocessor.inverse_transform(y_test, preprocessor.target_col)
    predictions_original = preprocessor.inverse_transform(
        predictions.flatten(), 
        preprocessor.target_col
    )
    
    # Tính metrics
    metrics = {
        'MAE': mean_absolute_error(y_test_original, predictions_original),
        'RMSE': np.sqrt(mean_squared_error(y_test_original, predictions_original)),
        'R2': r2_score(y_test_original, predictions_original)
    }
    
    print("\nEnsemble Model Performance:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    
    return ensemble, predictions_original, y_test_original, metrics

In [25]:
if __name__ == "__main__":
    # Load dữ liệu
    df = load_and_prepare_data('VietstockFinance_DBD_Bao-cao-tai-chinh_CSTC_20251217-132913.xlsx')
    
    print("Data Shape:", df.shape)
    print("\nFirst few rows:")
    print(df[['Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS)',
              'Tỷ suất lợi nhuận gộp biên',
              'Tỷ lệ lãi EBIT']].head())
    
    # Lựa chọn pipeline
    print("\n" + "="*60)
    print("CHOOSE LSTM PIPELINE")
    print("="*60)
    print("1. Simple LSTM")
    print("2. Bidirectional LSTM")
    print("3. ConvLSTM")
    print("4. Ensemble LSTM")
    
    choice = input("\nEnter your choice (1-4): ")
    
    if choice == '1':
        results = run_lstm_pipeline(df, model_type='simple')
    elif choice == '2':
        results = run_lstm_pipeline(df, model_type='bidirectional')
    elif choice == '3':
        results = run_lstm_pipeline(df, model_type='conv_lstm')
    elif choice == '4':
        ensemble_results = run_ensemble_pipeline(df)
    else:
        print("Invalid choice. Running simple LSTM...")
        results = run_lstm_pipeline(df, model_type='simple')
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*60)

FileNotFoundError: [Errno 2] No such file or directory: 'VietstockFinance_DBD_Bao-cao-tai-chinh_CSTC_20251217-132913.xlsx'

In [ ]:
def save_model_pipeline(model, preprocessor, filename='lstm_pipeline.pkl'):
    """Lưu toàn bộ pipeline"""
    import pickle
    import joblib
    
    pipeline = {
        'model': model,
        'preprocessor': preprocessor,
        'model_weights': model.get_weights()
    }
    
    # Lưu model architecture
    model.save(f'lstm_model_{filename}.keras')
    
    # Lưu preprocessor
    joblib.dump(preprocessor, f'preprocessor_{filename}.pkl')
    
    print(f"Model saved as lstm_model_{filename}.keras")
    print(f"Preprocessor saved as preprocessor_{filename}.pkl")

def load_model_pipeline(model_filename, preprocessor_filename):
    """Tải pipeline đã lưu"""
    from tensorflow.keras.models import load_model
    import joblib
    
    # Load model
    model = load_model(model_filename)
    
    # Load preprocessor
    preprocessor = joblib.load(preprocessor_filename)
    
    return model, preprocessor